# Sprint 8B Sequence-Context Encoder Runner

**Runner-only notebook.** Model, encoder, training, evaluation, plotting, and reporting logic stays in the repository under `src/`, `scripts/`, and `configs/`.

Execution plan: `docs/exec-plans/active/008b-sprint8b-sequence-context-encoder.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved code revision is pushed and Drive contains the raw data plus Sprint 3 / Sprint 5B processed graph artifacts. If Sprint 5B Graph C S5F2 is missing, this runner rebuilds it from Sprint 3 artifacts and raw data.

**This notebook prepares and calls the runner only.**  
Full canonical training belongs to Slice 5, not this notebook.  
No headline claims or result interpretation should be made from this notebook.

## Step 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Clone Or Update Repo Checkout

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/YasinEkici/crispr-gnn-offtarget.git}"
REPO_DIR="/content/crispr-gnn-offtarget"
GIT_REF="${GIT_REF:-sprint8/context-aware-interaction}"
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


## Step 3 - Dependency Sync And Runtime Check

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import torch
import torch_geometric
print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY


## Step 4 - Copy Drive Data And Processed Graph Artifacts

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found. Checked:" >&2
  printf '  %s\n' "${DRIVE_ROOT_CANDIDATES[@]}" >&2
  echo "Available MyDrive directories:" >&2
  find /content/drive/MyDrive -maxdepth 1 -type d | sort >&2
  exit 1
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
else
  echo "Missing $DRIVE_ROOT/data/raw; raw data is required if Sprint 5B Graph C must be rebuilt" >&2
  exit 1
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
else
  echo "Missing $DRIVE_ROOT/data/processed; Sprint 3/Sprint 5B graph artifacts are required" >&2
  exit 1
fi
if [ ! -d data/processed/graphs/sprint3/graph_c_context_observation ]; then
  echo "Missing local Sprint 3 Graph C artifacts after copy: data/processed/graphs/sprint3/graph_c_context_observation" >&2
  exit 1
fi
find data/processed/graphs -maxdepth 3 -type f -name 'manifest.json' | sort


## Step 5 - Build Or Validate Sprint 5B Graph C S5F2 Artifact

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ] || [ ! -f data/processed/graphs/sprint5b/graph_c_context_observation/features_S5F2_energy.parquet ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact required by Sprint 8B..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py \
    --data-config configs/data/mak2022.yaml \
    --schema-config configs/sweeps/graph_schema_ablation.yaml \
    --source-artifact-dir data/processed/graphs/sprint3 \
    --artifact-dir data/processed/graphs/sprint5b \
    --report-path outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md
fi
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_C
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
materialized = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5b')).load(GRAPH_C)
manifest = materialized.manifest
feature_tables = manifest.get('feature_tables', {})
if int(feature_tables.get('S5F2_energy', 0)) != 268:
    raise SystemExit(f'Missing 268-column Graph C S5F2_energy feature table: {feature_tables}')
if int(feature_tables.get('target_observation_features', 0)) != 212:
    raise SystemExit(f'Missing 212-column target_observation_features: {feature_tables}')
print('graph_name:', manifest.get('graph_name'))
print('split_id:', manifest.get('split_id'))
print('label_scheme:', manifest.get('label_scheme'))
print('feature_tables:', feature_tables)
PY


## Step 6 - Run Sprint 8B Sequence-Context Comparison

This cell runs the full canonical R1/R2 trained matrix plus the R0 reference row via the repo runner script. Full canonical training belongs to **Slice 5**; this notebook is the preparation and runner harness only.

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint8b_sequence_context_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint8b_sequence_context.py \
  --config configs/sweeps/sprint8b_sequence_context.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint8b_sequence_context_run_id.txt


## Step 6b - Smoke / Debug Path (optional)

Run a single canonical trained run with `--max-epochs 2` for fast validation of the end-to-end pipeline before committing GPU time. **Skip this cell for the full canonical run.** This is debug-only; do not use its outputs for any headline claim.

In [ ]:
%%bash
# SMOKE / DEBUG ONLY - skip for full canonical run (Step 6).
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint8b_smoke_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint8b_sequence_context.py \
  --config configs/sweeps/sprint8b_sequence_context.yaml \
  --max-epochs 2 \
  --run S8B_R1_sequence_only \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint8b_sequence_context_run_id.txt
echo "Smoke run complete (debug-only, not for headline claims)"


## Step 7 - Return Outputs To Drive

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
test -n "$DRIVE_ROOT"
RUN_BASENAME=$(cat /content/sprint8b_sequence_context_run_id.txt)
LOCAL_OUT="outputs/sprint8b"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
if [ -e "$RETURN_ROOT" ]; then
  echo "Output already exists in Drive: $RETURN_ROOT" >&2
  exit 1
fi
mkdir -p "$RETURN_ROOT"
rsync -a --exclude='model.pt' --exclude='.DS_Store' "$LOCAL_OUT/" "$RETURN_ROOT/"
echo "$RETURN_ROOT" > /content/sprint8b_sequence_context_return_root.txt
echo "Copied Sprint 8B outputs to $RETURN_ROOT"
find "$RETURN_ROOT" -maxdepth 3 -type f | sort | head -100


## Step 8 - Validate Output Contract

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
OUT="outputs/sprint8b"
# Consolidated outputs (Sprint 8B Section 9)
test -f "$OUT/sequence_context_comparison.csv"
test -f "$OUT/sequence_context_report.md"
test -f "$OUT/sequence_context_run_manifest.json"
test -f "$OUT/graph_artifact_provenance.json"
# Diagnostics directory
test -d "$OUT/diagnostics"
test -f "$OUT/diagnostics/sequence_context_threshold_metrics.csv"
test -f "$OUT/diagnostics/sequence_context_deltas.csv"
test -f "$OUT/diagnostics/sequence_context_training_history.csv"
test -f "$OUT/diagnostics/sequence_context_predictions.csv"
test -f "$OUT/diagnostics/sequence_context_score_deciles.csv"
test -f "$OUT/diagnostics/sequence_context_per_guide_score_summary.csv"
test -f "$OUT/diagnostics/sequence_context_sequence_input_audit.csv"
test -f "$OUT/diagnostics/sequence_context_parameter_counts.csv"
test -f "$OUT/diagnostics/sequence_context_attention_summary.csv"
test -f "$OUT/diagnostics/sequence_context_target_context_encoder_summary.csv"
test -f "$OUT/diagnostics/sequence_context_film_summary.csv"
# Figures directory
test -d "$OUT/figures"
test -f "$OUT/figures/sequence_context_auprc_comparison.png"
test -f "$OUT/figures/sequence_context_threshold_metrics.png"
test -f "$OUT/figures/sequence_context_pr_curves.png"
test -f "$OUT/figures/sequence_context_roc_curves.png"
test -f "$OUT/figures/sequence_context_score_distributions.png"
test -f "$OUT/figures/sequence_context_training_curves.png"
test -f "$OUT/figures/sequence_context_parameter_counts.png"
# Per-run artifacts for the canonical trained R1/R2 runs
RUN_PREFIX=$(cat /content/sprint8b_sequence_context_run_id.txt)
for RUN_ID in S8B_R1_sequence_only S8B_R2_sequence_plus_context; do
  RUN_DIR="$OUT/runs/${RUN_PREFIX}_${RUN_ID}"
  test -f "$RUN_DIR/resolved_config.yaml"
  test -f "$RUN_DIR/runtime.json"
  test -f "$RUN_DIR/training_history.csv"
  test -f "$RUN_DIR/metrics.csv"
  test -f "$RUN_DIR/sequence_input_audit.csv"
  test -f "$RUN_DIR/model.pt"
done
# Context/attention/FiLM summaries are expected only for the late-fusion R2 run.
R2_DIR="$OUT/runs/${RUN_PREFIX}_S8B_R2_sequence_plus_context"
test -f "$R2_DIR/attention_summary.csv"
test -f "$R2_DIR/target_context_encoder_summary.csv"
test -f "$R2_DIR/context_edge_interaction_summary.csv"
# Returned Drive copy must contain reportable outputs but exclude checkpoints.
RETURN_ROOT=$(cat /content/sprint8b_sequence_context_return_root.txt)
test -d "$RETURN_ROOT"
test -f "$RETURN_ROOT/sequence_context_comparison.csv"
test -f "$RETURN_ROOT/sequence_context_report.md"
test -f "$RETURN_ROOT/sequence_context_run_manifest.json"
test -f "$RETURN_ROOT/graph_artifact_provenance.json"
test -f "$RETURN_ROOT/diagnostics/sequence_context_threshold_metrics.csv"
test -f "$RETURN_ROOT/diagnostics/sequence_context_predictions.csv"
test -f "$RETURN_ROOT/diagnostics/sequence_context_sequence_input_audit.csv"
test -f "$RETURN_ROOT/figures/sequence_context_auprc_comparison.png"
if find "$RETURN_ROOT" -name 'model.pt' -print -quit | grep -q .; then
  echo "Returned Drive outputs must exclude model.pt checkpoints" >&2
  exit 1
fi
# Manifest run-ID validation and sequence-input audit checks.
PYTHONPATH=src uv run python - <<'PY'
import json
from pathlib import Path
import pandas as pd

manifest = json.loads(Path('outputs/sprint8b/sequence_context_run_manifest.json').read_text())
ids = {run['predeclared_id'] for run in manifest['runs']}
expected = {'S8B_R0_reference', 'S8B_R1_sequence_only', 'S8B_R2_sequence_plus_context'}
missing = expected - ids
if missing:
    raise SystemExit(f'Missing Sprint 8B run IDs in manifest: {sorted(missing)}')
if manifest.get('selection_metric') != 'validation_auprc':
    raise SystemExit(f"Unexpected selection metric: {manifest.get('selection_metric')}")

results = pd.read_csv('outputs/sprint8b/sequence_context_comparison.csv')
result_ids = set(results['predeclared_run_id'].astype(str))
missing_results = expected - result_ids
if missing_results:
    raise SystemExit(f'Missing Sprint 8B run IDs in comparison CSV: {sorted(missing_results)}')
headline = results.loc[results['predeclared_run_id'].isin(['S8B_R1_sequence_only', 'S8B_R2_sequence_plus_context'])]
if set(headline['sequence_context_mode']) != {'sequence_only', 'late_fusion'}:
    raise SystemExit('Sprint 8B canonical sequence modes drifted')
if not headline['external_pretrained_weights'].eq(False).all():
    raise SystemExit('Sprint 8B same-contract rows must not use external pretrained weights')

audit = pd.read_csv('outputs/sprint8b/diagnostics/sequence_context_sequence_input_audit.csv')
if set(audit['representation']) != {'S1_sequence_pair_from_graph_c_onehot'}:
    raise SystemExit('Unexpected Sprint 8B sequence representation')
if audit['guide_onehot_columns'].min() != 115 or audit['target_onehot_columns'].min() != 115:
    raise SystemExit('Sprint 8B S1 audit must see 115 guide and 115 target one-hot columns')
if set(audit['channels']) != {11}:
    raise SystemExit('Sprint 8B S1 audit must use 11 channels')

print('Sprint 8B output contract validated for batch:', manifest['batch_id'])
print()
cols = ['predeclared_run_id', 'sequence_context_mode', 'test_auprc', 'test_mcc', 'test_specificity', 'test_tn', 'test_fp']
available = [c for c in cols if c in results.columns]
print(results[available].to_string(index=False))
PY
